[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Relationships


## What you will be able to do

Give a hero and a team attributes that hold each other, with `Relationship` and `back_populates`,
and build a graph of objects without writing a single foreign key by hand. Read one side from the
other, and know when doing so sends another query. Decide what happens to the heroes when their team
is deleted, in Python with `cascade_delete` or in the database with `ondelete`, and see the
statements each one sends. Recognize the failures: the relationship with nothing to join on, the
misspelled name on the other side, two sides that quietly disagree, and the attribute read after its
session closed.


## The idea

### The problem

A hero has a `team_id` and a team has an id, and that is the whole of the connection so far. Every
question about it is a query written by hand: the heroes of a team is a `select` with a `where`, the
team of a hero is another `select`, and putting a new hero on a team means knowing the team's id
before the hero can be built. Nothing about that is hard; there is just a great deal of it, and it
spreads the shape of the data over every function that touches it.

Worse, a program that only has ids is easy to get half right. A hero is moved by setting `team_id`,
and a list of that team's heroes that was read a moment ago is now wrong, and nothing says so. The
two facts are the same fact, and the code holds them in two places.

Then there is the question nobody asks until it happens: what becomes of the heroes when a team is
deleted? There are three reasonable answers, and a program that has not chosen one will get whichever
its models happen to imply.

### What a relationship is

> A **`Relationship`** is an attribute that holds the objects on the other side of a foreign key.
> `Team.heroes` is a list of heroes, `Hero.team` is a team or `None`, and **`back_populates`** names
> the attribute on the other side so that the two are kept in step in memory. It stores nothing of
> its own: the foreign key column is still what the database holds, and SQLModel fills it in at the
> flush. Reading a relationship that has not been loaded sends a query, which is a **lazy load**,
> and it needs a session. **`cascade_delete=True`** tells the session to delete the children with
> the parent, **`ondelete="CASCADE"`** tells the database to do it, and **`passive_deletes=True`**
> tells the session to leave it to the database.

### Why it works that way

- **The foreign key is still the truth.** A relationship is a convenience over it, and assigning
  `hero.team = preventers` sets `hero.team_id` at the flush.
- **`back_populates` is what makes the two sides one fact.** Appending to `team.heroes` sets
  `hero.team` and the other way round. Without it both attributes exist, both write the same column,
  and neither knows about the other, which SQLAlchemy warns about.
- **A relationship is loaded when it is read.** That is convenient in a loop and expensive in a
  loop, which is the **Loading and N+1** notebook's whole subject, and it needs a live session,
  which is why an object read after its block raises.
- **Deleting has to be decided.** With nothing said, the session sets the children's foreign key to
  null. `cascade_delete` deletes them from Python, one statement each; `ondelete` deletes them in
  the database, in one; and the database's rule is the only one that also covers rows deleted by
  something that is not your program.

### Where this shows up

Every model that points at another, which is most of them. The **Many to Many** notebook does the
same for two tables that point at each other through a third, and the **Loading and N+1** notebook
is about when the queries a relationship sends become the problem. The **SQLAlchemy, Deep Dive**
guide's Relationships and Cascades and Deletes notebooks are the long version, with the cascade
rules in full.

### What this notebook covers

- What a relationship gives you, beside the query it replaces
- Building a graph, with no foreign key written by hand
- `back_populates`, and the two sides in step
- The query a relationship sends when it is read
- Deleting a team: what happens with nothing said
- `cascade_delete`, `ondelete` and who does the work
- Which rule to choose
- A team disbanded, finished
- Four failures, from a relationship with nothing to join on to an attribute read too late

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    team_id: int | None = Field(default=None, foreign_key="team.id")
    team: Team | None = Relationship(back_populates="heroes")


engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    preventers = Team(name="Preventers")
    session.add(Hero(name="Rusty-Man", team=preventers))            # no team_id anywhere
    session.commit()

    team = session.get(Team, 1)
    print("the team's heroes:", [hero.name for hero in team.heroes])
    print("the hero's team  :", team.heroes[0].team.name, "| team_id:", team.heroes[0].team_id)
```

```
the team's heroes: ['Rusty-Man']
the hero's team  : Preventers | team_id: 1
```

The hero was given a team object, not an id, and the team had no id to give when the hero was built.
At the commit the team was inserted, its id came back, and the hero's `team_id` was filled in with
it. Adding the hero added the team as well, because the session follows a relationship to whatever
is attached to it.


## Setup

Thirteen imports, one of them installed first where it is missing, the cast, four helpers, the
classes with their relationships, the engine, and the database built and loaded.

- `sqlmodel` is the library, and `SQLModel`, `Field`, `Relationship`, `Session`, `create_engine` and
  `select`, from it, are the classes, the attributes that hold each other, the session and the
  reading. Colab does not have SQLModel, so the cell installs 0.0.42 with `pip` where it is missing,
  and `version` and `PackageNotFoundError`, from `importlib.metadata`, `subprocess` and `sys` find
  out whether it is
- `event` and `insert`, from `sqlalchemy`, are the pragma on every connection and the rows `build`
  loads without a session, and `DetachedInstanceError`, from `sqlalchemy.orm.exc`, is what a
  relationship read after its session closed raises
- `logging` carries the SQL an engine logs to `PrintStatements`, which is how the queries a
  relationship sends are shown without the time in front of them
- `re` takes memory addresses out of a message, and `warnings` and `contextlib` make `catching`
- `subprocess` and `sys` also run `run_python`, which runs a file in a Python of its own: the last
  two Common errors are mistakes that break a mapper, and a mapper that fails to configure stays
  broken for the life of the process it happened in
- `Path` names the database file, and `shutil` removes the scratch folder at the start and at the end
- `TEAMS` and `HEROES` are the cast, which `build` loads

The classes now carry `Team.heroes` and `Hero.team`, and are the ones the rest of the guide uses. The
tables are exactly what they were: a relationship adds no column.


In [1]:
import contextlib
import logging
import re
import shutil
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import event, func, insert
from sqlalchemy.orm.exc import DetachedInstanceError
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


@contextlib.contextmanager
def catching():
    """Collects warnings instead of printing them: Python prints one with the file that raised it."""
    with warnings.catch_warnings(record=True) as raised:
        warnings.simplefilter("always")
        yield raised


class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again

def run_python(path):
    """Run a file in a Python of its own and print what it printed."""
    done = subprocess.run([sys.executable, path], capture_output=True, text=True)
    print(done.stdout.strip() or done.stderr.strip().splitlines()[-1])


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes in",
          len(session.exec(select(Team)).all()), "teams")


sqlmodel 0.0.42 | 8 heroes in 3 teams


## Worked examples

### What a relationship gives you, beside the query it replaces

Both sides, read as attributes, beside the queries they save:


In [2]:
with Session(engine) as session:
    preventers = session.get(Team, 1)
    print("team.heroes :", sorted(hero.name for hero in preventers.heroes))
    print("by query    :", sorted(hero.name for hero in
                                  session.exec(select(Hero).where(Hero.team_id == preventers.id))))

    deadpond = session.get(Hero, 1)
    print("hero.team   :", deadpond.team.name)
    print("by query    :", session.exec(select(Team).where(Team.id == deadpond.team_id)).one().name)


team.heroes : ['Captain North America', 'Rusty-Man', 'Spider-Boy', 'Tarantula']
by query    : ['Captain North America', 'Rusty-Man', 'Spider-Boy', 'Tarantula']
hero.team   : Z-Force
by query    : Z-Force


The same answers. What the attributes save is not the typing: it is having the shape of the data in
the classes instead of in every function that reads them.

### Building a graph, with no foreign key written by hand

A new team and two new heroes, connected by objects:


In [3]:
with Session(engine) as session:
    watch = Team(name="Night Watch", headquarters="The Bell Tower")
    owl = Hero(name="Night Owl", secret_name="Bruce Kent", age=41, team=watch)
    watch.heroes.append(Hero(name="Lady Ice", secret_name="Kara Frost", age=29))

    print("before the commit: team id", watch.id, "| owl team_id", owl.team_id)
    session.add(watch)                                              # the heroes come with it
    session.commit()
    session.refresh(watch)
    print("after the commit : team id", watch.id,
          "| heroes", sorted(hero.name for hero in watch.heroes))


before the commit: team id None | owl team_id None
after the commit : team id 4 | heroes ['Lady Ice', 'Night Owl']


Two ways to say the same thing, `team=watch` on one hero and `append` on the team, and neither
mentions an id. Only the team was added to the session, and both heroes were written with it,
because the session follows the relationship from whatever it is given.

### back_populates, and the two sides in step

`back_populates` is what makes an assignment on one side show on the other, in memory, before any
database is asked:


In [4]:
with Session(engine) as session:
    zforce = session.get(Team, 2)
    spider = session.get(Hero, 2)
    print("before:", spider.name, "is on", spider.team.name, "| Z-Force has",
          sorted(hero.name for hero in zforce.heroes))

    spider.team = zforce                                            # one assignment
    print("after :", spider.name, "is on", spider.team.name, "| Z-Force has",
          sorted(hero.name for hero in zforce.heroes))
    session.commit()


before: Spider-Boy is on Preventers | Z-Force has ['Black Lion', 'Deadpond', 'Dr. Weird']
after : Spider-Boy is on Z-Force | Z-Force has ['Black Lion', 'Deadpond', 'Dr. Weird', 'Spider-Boy']


The assignment was on the hero, and the team's list changed with it. That is `back_populates`: the
two attributes are two views of one fact, and SQLAlchemy keeps them together without going to the
database. The third of the Common errors is what happens when the two sides are not linked.

### The query a relationship sends when it is read

A relationship that has not been loaded is loaded when it is read, which is a query:


In [5]:
engine.echo = True
with Session(engine) as session:
    team = session.get(Team, 1)                                     # one query, for the team
    print("     the team is loaded")
    names = [hero.name for hero in team.heroes]                     # another, for its heroes
    print("     and now the heroes:", sorted(names))
engine.echo = False


    BEGIN (implicit)
    SELECT team.id AS team_id, team.name AS team_name, team.headquarters AS team_headquarters
    FROM team
    WHERE team.id = ?
    values: (1,)
     the team is loaded
    SELECT hero.id AS hero_id, hero.name AS hero_name, hero.secret_name AS hero_secret_name, hero.age AS hero_age, hero.team_id AS hero_team_id
    FROM hero
    WHERE ? = hero.team_id
    values: (1,)
     and now the heroes: ['Captain North America', 'Rusty-Man', 'Tarantula']
    ROLLBACK


Two statements: the `SELECT` for the team, then a second one for its heroes, sent the moment
`team.heroes` was read and not before. Reading it again in the same session sends nothing, because
the session already has them.

That is convenient once and expensive in a loop, which is the **Loading and N+1** notebook's
subject. It also needs a live session, which is where the last of the Common errors comes from.

### Deleting a team: what happens with nothing said

The classes so far say nothing about deleting. The heroes are not deleted, and they do not keep the
team either:


In [6]:
with Session(engine) as session:
    watch = session.exec(select(Team).where(Team.name == "Night Watch")).one()
    print("before:", sorted(hero.name for hero in watch.heroes))
    session.delete(watch)
    session.commit()

with Session(engine) as session:
    for hero in session.exec(select(Hero).where(Hero.name.in_(["Night Owl", "Lady Ice"]))):
        print("after :", hero.name, "| team_id", hero.team_id)


before: ['Lady Ice', 'Night Owl']
after : Lady Ice | team_id None
after : Night Owl | team_id None


No error, and two heroes on no team. The session found the children through the relationship and set
their foreign key to null, which is its default rule.

That is one of three reasonable answers, and it is worth noticing that it is a choice the models
made by not saying anything. The **Sessions** notebook deleted a team through classes that had no
relationship at all, and the database refused the delete outright, because nothing had told anybody
what to do with the heroes.

### cascade_delete, ondelete and who does the work

Saying it out loud takes two arguments: one for the session and one for the table. They go on a new
pair of classes rather than on `Team` and `Hero`, for a reason the last of the Common errors shows:
a class defined a second time leaves two of that name in SQLAlchemy's registry, and a relationship
that names its other side as text can no longer tell which is meant.


In [7]:
class Crew(SQLModel, table=True):
    """A team by another name, with the rule about deleting said out loud."""

    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)

    pilots: list["Pilot"] = Relationship(back_populates="crew", cascade_delete=True)


class Pilot(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    crew_id: int | None = Field(default=None, foreign_key="crew.id", ondelete="CASCADE")

    crew: Crew | None = Relationship(back_populates="pilots")


SQLModel.metadata.create_all(engine)
print("the foreign key:", [f"{key.column} {key.ondelete}" for key in Pilot.__table__.foreign_keys])

with Session(engine) as session:
    crew = Crew(name="Sky Patrol")
    crew.pilots.append(Pilot(name="Cloud Runner"))
    crew.pilots.append(Pilot(name="Storm Kite"))
    session.add(crew)
    session.commit()
    print("pilots:", len(session.exec(select(Pilot)).all()))


the foreign key: ['crew.id CASCADE']
pilots: 2


`ondelete="CASCADE"` is on the field, so it is in the table's foreign key, and `cascade_delete=True`
is on the relationship, so the session knows as well. Deleting the crew now takes the pilots with
it, and the statements say who did the work:


In [8]:
engine.echo = True
with Session(engine) as session:
    session.delete(session.get(Crew, 1))
    session.commit()
engine.echo = False

with Session(engine) as session:
    print("pilots left:", len(session.exec(select(Pilot)).all()))


    BEGIN (implicit)
    SELECT crew.id AS crew_id, crew.name AS crew_name
    FROM crew
    WHERE crew.id = ?
    values: (1,)
    SELECT pilot.id AS pilot_id, pilot.name AS pilot_name, pilot.crew_id AS pilot_crew_id
    FROM pilot
    WHERE ? = pilot.crew_id
    values: (1,)
    DELETE FROM pilot WHERE pilot.id = ?
    values: [(1,), (2,)]
    DELETE FROM crew WHERE crew.id = ?
    values: (1,)
    COMMIT
pilots left: 0


The session read the pilots of that crew, sent a `DELETE` for each of them, and then one for the
crew: three statements, and Python decided every one. `passive_deletes=True` on the relationship
says the database will do it instead. The same classes with that one argument added cannot be
defined again here, so they are run in a Python of their own:


In [9]:
%%writefile scratch/passive.py
import logging

from sqlalchemy import event
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select


class Crew(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)

    pilots: list["Pilot"] = Relationship(back_populates="crew", cascade_delete=True,
                                         passive_deletes=True)      # the database does the deleting


class Pilot(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    crew_id: int | None = Field(default=None, foreign_key="crew.id", ondelete="CASCADE")

    crew: Crew | None = Relationship(back_populates="pilots")


class PrintStatements(logging.Handler):
    def emit(self, record):
        if record.msg != "[%s] %r":
            print("   ", " ".join(record.getMessage().split()))


log = logging.getLogger("sqlalchemy.engine.Engine")
log.handlers = [PrintStatements()]
log.propagate = False

engine = create_engine("sqlite:///scratch/passive.db")


@event.listens_for(engine, "connect")
def keep_foreign_keys(connection, record):
    cursor = connection.cursor()
    cursor.execute("PRAGMA foreign_keys=ON")
    cursor.close()


SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    crew = Crew(name="Sky Patrol")
    crew.pilots.append(Pilot(name="Cloud Runner"))
    crew.pilots.append(Pilot(name="Storm Kite"))
    session.add(crew)
    session.commit()

engine.echo = True
with Session(engine) as session:
    session.delete(session.get(Crew, 1))
    session.commit()
engine.echo = False

with Session(engine) as session:
    print("pilots left:", len(session.exec(select(Pilot)).all()))


Writing scratch/passive.py


In [10]:
run_python("scratch/passive.py")


BEGIN (implicit)
    SELECT crew.id AS crew_id, crew.name AS crew_name FROM crew WHERE crew.id = ?
    DELETE FROM crew WHERE crew.id = ?
    COMMIT
pilots left: 0


One `DELETE`, for the crew alone, and no pilots left: the database followed `ON DELETE CASCADE` and
the session never read them. That is the difference the argument makes, and it is worth having where
a parent has many children, since the other way is one statement for each of them.

### Which rule to choose

| What should happen | Write | Who does it |
|---|---|---|
| the children lose the parent and stay | nothing; it is the default | the session, one `UPDATE` |
| the children go with the parent | `Relationship(cascade_delete=True)` and `Field(ondelete="CASCADE")` | the session, one `DELETE` each |
| the same, in one statement | add `passive_deletes=True` to the relationship | the database |
| the parent may not be deleted while children exist | leave the relationship off the parent, or `ondelete="RESTRICT"` | the database |

The two arguments are worth keeping together whichever you choose. The relationship's rule is the
only one the session follows, and the table's is the only one that covers a row deleted by a
migration, another service, or somebody at a command line.

### A team disbanded, finished

The pieces of this notebook in two functions. `move_hero` changes the side of a relationship, and
`disband` deletes a team and reports what went with it:


In [11]:
def move_hero(session, hero_name, team_name):
    """Put a hero on another team, by objects rather than by ids."""
    hero = session.exec(select(Hero).where(Hero.name == hero_name)).one()
    hero.team = session.exec(select(Team).where(Team.name == team_name)).one()
    session.commit()
    return f"{hero.name} is on the {hero.team.name}"


def disband(session, team_name):
    """Delete a team, and say which heroes it leaves without one."""
    team = session.exec(select(Team).where(Team.name == team_name)).one()
    losing = sorted(hero.name for hero in team.heroes)
    session.delete(team)
    session.commit()
    return {"disbanded": team_name, "heroes left without a team": losing}


with Session(engine) as session:
    print(move_hero(session, "Deadpond", "Preventers"))
    print(disband(session, "Z-Force"))
    print("heroes left:", sorted(hero.name for hero in session.exec(select(Hero))))


Deadpond is on the Preventers
{'disbanded': 'Z-Force', 'heroes left without a team': ['Black Lion', 'Dr. Weird', 'Spider-Boy']}
heroes left: ['Black Lion', 'Captain North America', 'Deadpond', 'Dr. Weird', 'Lady Ice', 'Night Owl', 'Princess Sure-E', 'Rusty-Man', 'Spider-Boy', 'Tarantula']


`move_hero` never mentions a foreign key, and `disband` could list the heroes it was about to affect
because reading `team.heroes` loaded them. `Team` says nothing about deleting, so those heroes are
still here with no team at all, and the count of heroes is unchanged: the rule that would have taken
them with it is the one `Crew` and `Pilot` carry.

### Where each part came from

| In `move_hero` and `disband` | What it relies on | The section that showed it |
|---|---|---|
| `hero.team = ...` | an assignment that sets the foreign key at the flush | Building a graph |
| `hero.team.name` after the assignment | the two sides kept in step by `back_populates` | `back_populates`, and the two sides in step |
| `team.heroes` before the delete | a relationship loaded when it is read | The query a relationship sends |
| `session.delete(team)` leaving the heroes | the default rule, which sets their foreign key to null | Deleting a team: what happens with nothing said |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/08-relationships-solutions.ipynb).

**1.** Print every team with the names of its heroes, using the relationship and no `select` for the
heroes.


In [12]:
# your code here


**2.** Make a team called Sky Patrol at Cloud Base with two heroes of your own, adding only the team
to the session, and print the ids the database gave all three.


In [13]:
# your code here


**3.** Move one of those heroes to the Preventers by assigning `hero.team`, and print both teams'
hero lists before and after, in the same session.


In [14]:
# your code here


**4.** Print the statements that reading `team.heroes` sends for a team the session has just loaded,
with `echo`, and say in a comment how many there are.


In [15]:
# your code here


**5.** Count the heroes on each team without touching the relationship, then with it, and show the
two agree.


In [16]:
# your code here


**6.** Disband Sky Patrol and show what became of its heroes, then print how many heroes there are.


In [17]:
# your code here


## Common errors

### sqlalchemy.orm.exc.DetachedInstanceError: Parent instance <Hero at 0x...> is not bound to a Session; lazy load operation of attribute 'team' cannot proceed


In [18]:
with Session(engine) as session:
    hero = session.get(Hero, 1)
    print("inside the block:", hero.name)

try:
    print(hero.team)
except DetachedInstanceError as error:                              # its message names a memory address
    print(type(error).__name__ + ":", message(error))


inside the block: Deadpond
DetachedInstanceError: Parent instance <Hero at 0x...> is not bound to a Session; lazy load operation of attribute 'team' cannot proceed (Background on this error at: https://sqlalche.me/e/20/bhk3)


The hero's own columns were loaded and are still readable; the team was never loaded, and loading it
now would take a query, and there is no session to send one. The message says which attribute it was,
which is the useful part.

Three answers, in order of how often they are right. Read what you need inside the block. Load the
relationship on purpose while the session is open, with `selectinload`, which the
**Loading and N+1** notebook is about. Or turn the objects into models with no table before the block
ends, which is what the **Create, Read and Update Models** notebook's public models are for:


In [19]:
with Session(engine) as session:
    hero = session.get(Hero, 1)
    team_name = hero.team.name if hero.team else None               # read it while there is a session

print("read inside the block:", team_name)


read inside the block: Preventers


### No error, and two objects that disagree: a relationship with no back_populates


In [20]:
with catching() as warned:

    class Squad(SQLModel, table=True):
        id: int | None = Field(default=None, primary_key=True)
        name: str = Field(max_length=50)

        members: list["Member"] = Relationship()                    # no back_populates

    class Member(SQLModel, table=True):
        id: int | None = Field(default=None, primary_key=True)
        name: str = Field(max_length=50)
        squad_id: int | None = Field(default=None, foreign_key="squad.id")

        squad: Squad | None = Relationship()                        # and none here either

    SQLModel.metadata.create_all(engine)
    with Session(engine) as session:
        squad = Squad(name="Sky Patrol")
        member = Member(name="Cloud Runner")
        squad.members.append(member)
        print("after the append, member.squad is:", member.squad)
        session.add(squad)
        session.commit()
        session.refresh(member)
        print("after the commit, squad_id is   :", member.squad_id)

print("warned:", [str(warning.message)[:96] for warning in warned])


after the append, member.squad is: None
after the commit, squad_id is   : 1
warned: ["relationship 'Member.squad' will copy column squad.id to column member.squad_id, which conflicts"]


The member was appended to the squad and does not know it: `member.squad` is `None` until the commit
writes the column and the refresh reads it back. Between those two moments any code that asked the
member which squad it was on got the wrong answer, and nothing raised.

SQLAlchemy warns about the cause, in a message worth reading twice: two relationships copy the same
column and neither is linked to the other. `back_populates` on both sides is the fix, which is what
`Team` and `Hero` have.

### sqlalchemy.exc.NoForeignKeysError: Could not determine join condition between parent/child tables on relationship Squad.members - there are no foreign keys linking these tables.


In [21]:
%%writefile scratch/no_foreign_key.py
from sqlalchemy.orm import configure_mappers
from sqlmodel import Field, Relationship, SQLModel


class Squad(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    members: list["Member"] = Relationship(back_populates="squad")


class Member(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    squad: Squad | None = Relationship(back_populates="members")    # nothing points at squad.id


try:
    configure_mappers()
except Exception as error:
    print(type(error).__module__ + "." + type(error).__name__)
    print(str(error).splitlines()[0])


Writing scratch/no_foreign_key.py


In [22]:
run_python("scratch/no_foreign_key.py")


sqlalchemy.exc.NoForeignKeysError
Could not determine join condition between parent/child tables on relationship Squad.members - there are no foreign keys linking these tables.  Ensure that referencing columns are associated with a ForeignKey or ForeignKeyConstraint, or specify a 'primaryjoin' expression.


A relationship is an attribute over a foreign key, and there is no foreign key here: `Member` has no
`squad_id`. The message names the relationship it could not work out and says what is missing.

The error arrives when the mappers are configured, which happens the first time any model is used
rather than when the class is defined, so a program can start and fail at its first query. That is
also why this one runs in a Python of its own: a mapper that fails to configure stays broken for the
rest of the process, and every later query raises `One or more mappers failed to initialize`
instead of anything useful. In a notebook the cure is to correct the class and restart the runtime.

The fix is the field the relationship needs:


In [23]:
%%writefile scratch/with_foreign_key.py
from sqlalchemy.orm import configure_mappers
from sqlmodel import Field, Relationship, SQLModel


class Squad(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    members: list["Member"] = Relationship(back_populates="squad")


class Member(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    squad_id: int | None = Field(default=None, foreign_key="squad.id")
    squad: Squad | None = Relationship(back_populates="members")


configure_mappers()
print("the mappers configured, and the join is", Squad.members.property.primaryjoin)


Writing scratch/with_foreign_key.py


In [24]:
run_python("scratch/with_foreign_key.py")


the mappers configured, and the join is squad.id = member.squad_id


### sqlalchemy.exc.InvalidRequestError: Mapper 'Mapper[Member(member)]' has no property 'squadd'.


In [25]:
%%writefile scratch/misspelled.py
from sqlalchemy.orm import configure_mappers
from sqlmodel import Field, Relationship, SQLModel


class Squad(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    members: list["Member"] = Relationship(back_populates="squadd")     # the attribute is squad


class Member(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    squad_id: int | None = Field(default=None, foreign_key="squad.id")
    squad: Squad | None = Relationship(back_populates="members")


try:
    configure_mappers()
except Exception as error:
    print(type(error).__module__ + "." + type(error).__name__)
    print(str(error).splitlines()[0])


Writing scratch/misspelled.py


In [26]:
run_python("scratch/misspelled.py")


sqlalchemy.exc.InvalidRequestError
Mapper 'Mapper[Member(member)]' has no property 'squadd'.  If this property was indicated from other mappers or configure events, ensure registry.configure() has been called.


`back_populates` is the name of the attribute on the other class, written as text, so nothing checks
the spelling until the mappers are configured. The message is precise: the mapper for `Member` has no
property with that name.

This is the loud half of the earlier silent one. A misspelled `back_populates` raises; leaving it off
both sides raises nothing and quietly lets the two attributes disagree.

### sqlalchemy.exc.InvalidRequestError: Multiple classes found for path "Hero" in the registry of this declarative base.


In [27]:
with catching() as warned:                                          # a model cell, run a second time
    SQLModel.metadata.clear()

    class Team(SQLModel, table=True):
        id: int | None = Field(default=None, primary_key=True)
        name: str = Field(index=True, max_length=50)
        headquarters: str = Field(max_length=60)

        heroes: list["Hero"] = Relationship(back_populates="team")

    class Hero(SQLModel, table=True):
        id: int | None = Field(default=None, primary_key=True)
        name: str = Field(index=True, max_length=50)
        secret_name: str = Field(max_length=60)
        age: int | None = Field(default=None, index=True)
        team_id: int | None = Field(default=None, foreign_key="team.id")

        team: Team | None = Relationship(back_populates="heroes")

print("defined again, and the metadata holds", sorted(SQLModel.metadata.tables))
with Session(engine) as session:
    session.exec(select(Hero)).first()


defined again, and the metadata holds ['hero', 'team']


InvalidRequestError: Multiple classes found for path "Hero" in the registry of this declarative base. Please use a fully module-qualified path.

`SQLModel.metadata.clear()` empties the tables, and it does not empty SQLAlchemy's registry of
classes. After a second definition that registry holds two classes called `Hero`, and `list["Hero"]`
is a name to look up, so the first query that configures the mappers cannot tell which of the two
the relationship means.

This is the most common way to meet it: running a model cell twice in a notebook. It is why the
**table=True** notebook's advice, to empty the metadata and run every model cell again, stops being
enough once the classes have relationships in them, and why this notebook gave the cascade rules to
a new pair of classes rather than writing `Team` and `Hero` a second time. Restarting the runtime is
the way back, since nothing in a running process takes a class out of that registry.

Last, the engine lets go of the file, and this cell removes the scratch folder with the database and
the files in it:


In [28]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `Relationship` is an attribute over a foreign key: `Team.heroes` is a list, `Hero.team` is one team
  or `None`, and neither adds a column.
- `back_populates` names the attribute on the other side, and is what keeps the two in step in
  memory; without it on both sides they disagree until a commit and a reload.
- Assigning an object, or appending to a list, is enough: the foreign key is filled in at the flush,
  and adding one object adds what is attached to it.
- Reading a relationship that is not loaded sends a query, so it needs a live session and raises
  `DetachedInstanceError` after the block.
- A class defined a second time stays in SQLAlchemy's registry beside the first, so a relationship
  that names its other side as text can no longer be resolved: restart the runtime instead.
- With nothing said, deleting a parent sets the children's foreign key to null; `cascade_delete=True`
  deletes them from Python, `ondelete="CASCADE"` from the database, and `passive_deletes=True` leaves
  the work to the database.


## What is next

The **Many to Many** notebook connects two tables that both point at many of the other: a link model
with a primary key of two columns, `link_model` on the relationship, the extra column on the link
that `append` has no way to fill in, and the two foreign keys to one table that SQLAlchemy cannot
choose between.


---

&#8592; **Previous:** [Create, Read and Update Models](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/07-create-read-and-update-models.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Many to Many](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/09-many-to-many.ipynb) &#8594;
